# SVG Chinese converter
Local OpenCC batch conversion. Fonts and span boundaries are preserved. Run all cells from the project directory.

## 1. Configuration

In [ ]:
from pathlib import Path
from svg_chinese import convert_svg, output_paths

CONVERSION_DIRECTION = "s2t"  # "s2t" or "t2s"
INPUT_DIR = Path("inputs")


## 2. Discover sources
Select only SVG filenames containing the source language designation, in sorted filename order.

In [ ]:
if CONVERSION_DIRECTION not in ("s2t", "t2s"):
    raise ValueError('Direction must be "s2t" or "t2s".')
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")
source_language = ("simplified_chinese" if CONVERSION_DIRECTION == "s2t"
                   else "traditional_chinese")
input_files = sorted(
    (p for p in INPUT_DIR.glob("*.svg") if p.is_file() and source_language in p.stem),
    key=lambda p: p.name,
)
print(f"Found {len(input_files)} matching SVGs ({CONVERSION_DIRECTION}).")


## 3. Convert pending files
Existing outputs are skipped. Existing reports are also protected. Each file uses the unchanged conversion and validation helper; a failure does not stop the batch.

In [ ]:
batch_results = []
for input_file in input_files:
    result = {"filename": input_file.name, "status": "failed", "warnings": ""}
    try:
        output_file, report_file = output_paths(input_file, CONVERSION_DIRECTION)
        if output_file.exists():
            result["status"] = "skipped"
            if not report_file.exists():
                result["warnings"] = "Output exists but QC report is missing; conversion skipped."
        elif report_file.exists():
            result.update(status="skipped", warnings="QC report exists without output; skipped to preserve report.")
        else:
            summary = convert_svg(input_file, CONVERSION_DIRECTION)
            result.update(status="converted", summary=summary)
            warnings = []
            if summary["segments_with_warnings"]:
                warnings.append(f"{summary['segments_with_warnings']} segments with QC warnings; see CSV")
            if summary["conversion_failures"]:
                warnings.append(f"{summary['conversion_failures']} segment conversion failures; originals retained")
            result["warnings"] = "; ".join(warnings)
    except Exception as exc:
        result["warnings"] = f"{type(exc).__name__}: {exc}"
    batch_results.append(result)


## 4. Validate and summarize
New outputs are reparsed and structurally checked by the helper. Skipped files are not revalidated or modified. Detailed QC remains in each CSV.

In [ ]:
from IPython.display import HTML, display
from html import escape

rows = []
for result in batch_results:
    rows.append("<tr>" + "".join(
        f"<td>{escape(result[key])}</td>" for key in ("filename", "status", "warnings")
    ) + "</tr>")
display(HTML("<table><tr><th>Filename</th><th>Status</th><th>Warnings / errors</th></tr>"
             + "".join(rows) + "</table>"))
print(" | ".join(f"{status}: {sum(r['status'] == status for r in batch_results)}"
                 for status in ("converted", "skipped", "failed")))


## 5. Output files
Review new QC reports and check fonts and layout manually in Adobe Illustrator. No font correction is performed. To reconvert a file, manually move both its existing output SVG and QC report out of these directories, then rerun.

In [ ]:
from IPython.display import FileLinks

display(FileLinks("outputs", recursive=False))
display(FileLinks("reports", recursive=False))
